# 📋 Planner-Executor Pattern

## What is Planner-Executor?
Two specialized agents:
1. **Planner**: breaks complex task into steps
2. **Executor**: executes each step one by one

## Why This Pattern?
Complex tasks need planning before execution.
Trying to do everything in one LLM call leads to:
- Incomplete solutions
- Missing edge cases
- Poor structure

## Mental Model: Architect + Builder
- Architect (Planner): creates the blueprint
- Builder (Executor): follows the blueprint


In [ ]:
# ── Planner-Executor Agent ─────────────────────────────────────────────
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field

class ExecutionPlan(BaseModel):
    steps: list[str] = Field(description='Ordered list of concrete steps to execute')
    goal: str = Field(description='The final goal of the plan')

class PlannerState(TypedDict):
    task: str
    plan: list[str]
    goal: str
    current_step: int
    results: list[str]

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
planner_llm = llm.with_structured_output(ExecutionPlan)

def planner(state: PlannerState) -> dict:
    plan = planner_llm.invoke([
        SystemMessage(content='Create a concrete step-by-step plan. Each step should be actionable.'),
        HumanMessage(content=f'Task: {state["task"]}')
    ])
    print(f'Plan created: {len(plan.steps)} steps')
    for i, step in enumerate(plan.steps, 1):
        print(f'  Step {i}: {step}')
    return {'plan': plan.steps, 'goal': plan.goal, 'current_step': 0}

def executor(state: PlannerState) -> dict:
    step_idx = state['current_step']
    current_step = state['plan'][step_idx]
    context = f'Previous results: {state["results"]}' if state['results'] else ''
    response = llm.invoke([
        SystemMessage(content=f'Execute this step for the goal: {state["goal"]}. {context}'),
        HumanMessage(content=f'Execute: {current_step}')
    ])
    print(f'Step {step_idx+1} done: {response.content[:60]}...')
    return {
        'results': state['results'] + [response.content],
        'current_step': step_idx + 1
    }

def has_more_steps(state: PlannerState) -> str:
    if state['current_step'] < len(state['plan']):
        return 'executor'
    return '__end__'

graph = StateGraph(PlannerState)
graph.add_node('planner', planner)
graph.add_node('executor', executor)
graph.set_entry_point('planner')
graph.add_edge('planner', 'executor')
graph.add_conditional_edges('executor', has_more_steps)
app = graph.compile()

result = app.invoke({
    'task': 'Explain how to make a Python web scraper',
    'plan': [], 'goal': '', 'current_step': 0, 'results': []
})
print(f'\nCompleted {len(result["results"])} steps')

## 🎯 Interview Questions

1. **What is the planner-executor pattern?**
2. **Why separate planning from execution?**
3. **How does the executor use results from previous steps?**
4. **When would you re-plan during execution?**

> Answer these before moving to the next module.

## 💪 Mini Exercise

**Exercise**: Based on what you learned in this module:

1. Re-run all code cells with your own inputs
2. Modify one code cell to add new functionality
3. Answer the interview questions above from memory

**Mini Project**: Build a small application that combines the concepts from this module.


## 📚 Summary

In this module you learned:
- The core concepts of **Planner-Executor Pattern — Structured Problem Solving**
- How to implement them with modern LangChain/LangGraph APIs
- Common mistakes and how to avoid them
- Interview-level understanding of why each component exists

**Next**: Continue to the next module to build on these foundations.

---
*Part of the LangChain & LangGraph Master Course*
